In [1]:
import openpyxl
import boto3
import os

In [ ]:
!date
jupyter_dir = os.getcwd()
aws_profile = "hogehoge"
print(f"DIR: {jupyter_dir}")
print(f"AWS profile: {aws_profile}")

In [3]:
# Excelファイルの読み込み
excel_file_pass = "./CloudWatchロググループ一覧.xlsx"
print(excel_file_pass + ' を読み込みます。')

wb = openpyxl.load_workbook(excel_file_pass)

In [ ]:
# シート一覧
for sheet in wb:
    print(sheet.title)

In [5]:
# シートの選択
ws = wb['dev環境']

In [6]:
# シートの初期化
for row in ws:
  for cell in row:
      cell.value = None


In [7]:
# ヘッダー行の追記
header = ['ロググループ名', '保持期間', 'メトリクスフィルター名', 'メトリクスフィルターパターン',
          'サブスクリプションフィルター名', 'サブスクリプションフィルターパターン']

for i in range(len(header)):
    ws.cell(1, i+1).value = header[i]

ws.freeze_panes = 'A2'

In [8]:
# AWS CloudWatch Logsクライアントの作成
my_session = boto3.Session(profile_name=aws_profile)
client = my_session.client('logs')

In [ ]:
# CloudWatch Logsのロググループ一覧を取得
name_pattern = 'fugafuga'
print(name_pattern + ' の文字列を含むロググループを抽出します。')


response = client.describe_log_groups(logGroupNamePattern=name_pattern)

row_num = 2
for log_group in response['logGroups']:
    log_group_name = log_group['logGroupName']

    # 保持期間を取得
    retention_period = client.describe_log_groups(logGroupNamePrefix=log_group_name)['logGroups'][0].get('retentionInDays', '-')

    # メトリクスフィルター一覧を取得
    metric_filters = client.describe_metric_filters(logGroupName=log_group_name).get('metricFilters', [])
    metric_filter_name = '\n'.join([filter['filterName'] for filter in metric_filters])
    metric_filter_pattern = '\n'.join([filter['filterPattern'] for filter in metric_filters])

    # サブスクリプションフィルター一覧を取得
    subscription_filters = client.describe_subscription_filters(logGroupName=log_group_name).get('subscriptionFilters', [])
    subscription_filter_name = '\n'.join([filter['filterName'] for filter in subscription_filters])
    subscription_filter_pattern = '\n'.join([filter['filterPattern'] for filter in subscription_filters])

    # Excelに行を追加
    rows = [log_group_name, retention_period, metric_filter_name, metric_filter_pattern,
           subscription_filter_name, subscription_filter_pattern]
    
    for i in range(len(rows)):
        sheet.cell(row=row_num, column=i+1, value=rows[i])

    row_num += 1

In [10]:
# 折り返して全体を表示
columns_to_wrap = [4, 6]  # D列とF列

# 各列に対して折り返しを有効にする
for col_num in columns_to_wrap:
    for row_num in range(1, ws.max_row + 1):
        cell = sheet.cell(row=row_num, column=col_num)
        cell.alignment = openpyxl.styles.Alignment(wrap_text=True)


In [ ]:
# Excelファイルを保存
wb.save(excel_file_pass)
print(excel_file_pass + ' に保存しました。')